In [ ]:
import pickle
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import t


## Settings

In [ ]:
# Set display options for pandas
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

In [ ]:
# Define dataset variable configurations
DATASET_VARS = {
    "BH_1": {
        "obj_var": "yield",
        "cat_vars": [
            "Aryl_halide_SMILES",
            "Additive_SMILES",
            "Base_SMILES",
            "Ligand_SMILES",
        ],
        "con_vars": [],
    },
    "DA": {
        "obj_var": "yield",
        "cat_vars": ["Base_SMILES", "Ligand_SMILES", "Solvent_SMILES"],
        "con_vars": ["Concentration", "Temp_C"],
    },
    "alkox": {
        "obj_var": "conversion",
        "cat_vars": [],
        "con_vars": ["catalase", "peroxidase", "alcohol_oxidase", "ph"],
    },
    "oer_plate_a": {
        "obj_var": "overpotential",
        "cat_vars": [],
        "con_vars": ["ni_load", "fe_load", "co_load", "mn_load", "ce_load", "la_load"],
    },
    "p3ht": {
        "obj_var": "conductivity",
        "cat_vars": [],
        "con_vars": [
            "p3ht_content",
            "d1_content",
            "d2_content",
            "d6_content",
            "d8_content",
        ],
    },
    "photo_pce10": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "photo_wf3": {
        "obj_var": "degradation",
        "cat_vars": [],
        "con_vars": ["mat_1", "mat_2", "mat_3", "mat_4"],
    },
    "suzuki_edbo": {
        "obj_var": "yield",
        "cat_vars": ["electrophile", "nucleophile", "base", "ligand", "solvent"],
        "con_vars": [],
    },
    "suzuki": {
        "obj_var": "yield",
        "cat_vars": [],
        "con_vars": ["temperature", "pd_mol", "arbpin", "k3po4"],
    },
}

In [ ]:
DATASET_NAME_ORDER = [
    "BH_1",
    "DA",
    "alkox",
    "oer_plate_a",
    "p3ht",
    "photo_pce10",
    "photo_wf3",
    "suzuki_edbo",
    "suzuki",
]
MODEL_ORDER = [
    "gpt-5-mini-2025-08-07",
    "o4-mini-2025-04-16",
    "gpt-4.1-mini-2025-04-14",
    "gpt-4o-mini-2024-07-18",
    "claude-sonnet-4-5-20250929",
    "claude-haiku-4-5-20251001",
    "claude-3-5-haiku-20241022",
]
MODEL_LABELS = {
    "gpt-5-mini-2025-08-07": "GPT-5 mini",
    "o4-mini-2025-04-16": "o4 mini",
    "gpt-4.1-mini-2025-04-14": "GPT-4.1 mini",
    "gpt-4o-mini-2024-07-18": "GPT-4o mini",
    "claude-sonnet-4-5-20250929": "Claude Sonnet 4.5",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    "claude-3-5-haiku-20241022": "Claude 3.5 Haiku",
}

In [ ]:
TIME_TAGS = [
    "20260222173044",
    "20260222173136",
    "20260222173220",
    "20260222173343",
    "20260222173455",
]

In [ ]:
REPEAT_NUM = 5

In [ ]:
# Read and concatenate batch output logs
batch_output_logs_df_list = []
for time_tag in TIME_TAGS:
    batch_output_logs_df_tmp = pd.read_csv(
        Path("./results_final") / f"results_{time_tag}" / "batch_output_logs.csv"
    )
    batch_output_logs_df_tmp.insert(2, "time_tag", time_tag)
    batch_output_logs_df_list.append(batch_output_logs_df_tmp)
batch_output_logs_df = pd.concat(batch_output_logs_df_list, axis=0, ignore_index=True)

# Sort by dataset_name and model
batch_output_logs_df["dataset_name"] = pd.Categorical(
    batch_output_logs_df["dataset_name"], categories=DATASET_NAME_ORDER, ordered=True
)
batch_output_logs_df["model"] = pd.Categorical(
    batch_output_logs_df["model"], categories=MODEL_ORDER, ordered=True
)
batch_output_logs_df = batch_output_logs_df.sort_values(
    ["dataset_name", "model"]
).reset_index(drop=True)

# Convert to string type
batch_output_logs_df["dataset_name"] = batch_output_logs_df["dataset_name"].astype(str)
batch_output_logs_df["model"] = batch_output_logs_df["model"].astype(str)

# Create a new column containing model parameters
batch_output_logs_df["model_w_params"] = batch_output_logs_df["model"].map(MODEL_LABELS)
for idx, row in batch_output_logs_df.iterrows():
    if pd.notna(batch_output_logs_df.loc[idx, "reasoning_effort"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += (
            " " + batch_output_logs_df.loc[idx, "reasoning_effort"]
        )
    elif pd.notna(batch_output_logs_df.loc[idx, "temperature"]):
        batch_output_logs_df.loc[idx, "model_w_params"] += " temp" + str(
            round(batch_output_logs_df.loc[idx, "temperature"], 2)
        )

# Save the summarized batch output logs
batch_output_logs_df.to_csv("results_final/batch_output_logs.csv", index=False)

## Scores

In [ ]:
batch_output_logs_df_wo_duplicate = (
    batch_output_logs_df[["dataset_name", "model", "model_w_params", "time_tag"]]
    .drop_duplicates()
    .reset_index(drop=True)
)
display(batch_output_logs_df_wo_duplicate)

In [ ]:
pair_result_dfs = {}

for i, batch_output_logs_df_row in batch_output_logs_df_wo_duplicate[
    ["dataset_name", "model", "model_w_params", "time_tag"]
].iterrows():
    dataset_name = batch_output_logs_df_row.dataset_name
    model = batch_output_logs_df_row.model
    model_w_params = batch_output_logs_df_row.model_w_params
    time_tag = batch_output_logs_df_row.time_tag

    print(f"dataset: {dataset_name}, model {model_w_params}")

    # Read experimental data
    exp_original_df = pd.read_csv(
        f"./dataset_processed/dataset_{dataset_name}_extracted.csv"
    )

    # Read original pair data
    pair_original_df = pd.read_csv(
        f"./dataset_processed/dataset_{dataset_name}_pair.csv"
    )
    obj_var = DATASET_VARS[dataset_name]["obj_var"]

    # Align judgment with LLM
    pair_original_df["setup_true"] = ""
    if dataset_name in [
        "oer_plate_a",
        "photo_pce10",
        "photo_wf3",
    ]:  # Lower is better
        pair_original_df.loc[
            pair_original_df[f"{obj_var}_A"] > pair_original_df[f"{obj_var}_B"],
            "setup_true",
        ] = "B"
        pair_original_df.loc[
            pair_original_df[f"{obj_var}_A"] < pair_original_df[f"{obj_var}_B"],
            "setup_true",
        ] = "A"
    else:
        pair_original_df.loc[
            pair_original_df[f"{obj_var}_A"] > pair_original_df[f"{obj_var}_B"],
            "setup_true",
        ] = "A"
        pair_original_df.loc[
            pair_original_df[f"{obj_var}_A"] < pair_original_df[f"{obj_var}_B"],
            "setup_true",
        ] = "B"
    assert len(pair_original_df.loc[pair_original_df["setup_true"] == ""]) == 0

    # Read pair prediction results
    pair_result_merged_df = pair_original_df.copy()
    repeat_forward_cols = []
    repeat_reverse_cols = []
    for repeat in range(REPEAT_NUM):
        for forward_or_reverse in ["forward", "reverse"]:
            pair_result_df = pd.read_csv(
                Path("./results_final")
                / f"results_{time_tag}"
                / f"dataset_{dataset_name}_{model}_rep{repeat + 1}_{forward_or_reverse}_pair_result.csv"
            )
            setup_renamed = f"setup_predicted_{forward_or_reverse}_{repeat + 1}"
            pair_result_df = pair_result_df.rename(
                columns={"answer_token": setup_renamed}
            )
            pair_result_df = pair_result_df[["ID_A", "ID_B", setup_renamed]]

            # Merge results (reverse pair prediction results swap ID_A and ID_B columns for merging)
            if forward_or_reverse == "reverse":
                pair_result_df = pair_result_df.rename(
                    columns={"ID_A": "ID_B", "ID_B": "ID_A"}
                )
            pair_result_merged_df = pd.merge(
                pair_result_merged_df, pair_result_df, on=["ID_A", "ID_B"], how="left"
            )

            if forward_or_reverse == "forward":
                repeat_forward_cols.append(setup_renamed)
            else:
                repeat_reverse_cols.append(setup_renamed)

    # Remove rows with NaN values
    pair_result_merged_df = pair_result_merged_df.dropna(
        subset=repeat_forward_cols + repeat_reverse_cols, how="any", axis=0
    )

    # Abstention rate
    pair_result_merged_df["abstention_count"] = (
        pair_result_merged_df[repeat_forward_cols] == "C"
    ).sum(axis=1) + (pair_result_merged_df[repeat_reverse_cols] == "C").sum(axis=1)
    pair_result_merged_df["abstention_rate"] = pair_result_merged_df[
        "abstention_count"
    ] / (REPEAT_NUM * 2)

    # Accuracy
    def calc_accuracy(row):
        forward_correct = 0
        reverse_correct = 0
        answer_A = 0
        answer_B = 0
        for f_col, r_col in zip(repeat_forward_cols, repeat_reverse_cols):
            # forward: if setup_true is A, then A is correct; if B, then B is correct
            if (
                row["setup_true"] == "A"
                and row[f_col] == "A"
                or row["setup_true"] == "B"
                and row[f_col] == "B"
            ):
                forward_correct += 1
            if row[f_col] == "A":
                answer_A += 1
            elif row[f_col] == "B":
                answer_B += 1
            # reverse: if setup_true is A, then B is correct; if B, then A is correct
            if (
                row["setup_true"] == "A"
                and row[r_col] == "B"
                or row["setup_true"] == "B"
                and row[r_col] == "A"
            ):
                reverse_correct += 1
            if row[r_col] == "A":
                answer_B += 1
            elif row[r_col] == "B":
                answer_A += 1
        correct_count = forward_correct + reverse_correct
        total_trials = REPEAT_NUM * 2 - row["abstention_count"]
        if total_trials == 0:
            overall_accuracy = 0.0
            answered_accuracy = np.nan
            majority_answer = "C"
            majority_answer_correct = np.nan
        else:
            overall_accuracy = correct_count / (REPEAT_NUM * 2)
            answered_accuracy = correct_count / total_trials
            majority_answer = (
                "A" if answer_A > answer_B else ("B" if answer_B > answer_A else "Tie")
            )
            majority_answer_correct = (
                1
                if majority_answer == row["setup_true"]
                else (0.5 if majority_answer == "Tie" else 0)
            )
        return (
            overall_accuracy,
            answered_accuracy,
            majority_answer,
            majority_answer_correct,
        )

    (
        pair_result_merged_df["overall_accuracy"],
        pair_result_merged_df["answered_accuracy"],
        pair_result_merged_df["majority_answer"],
        pair_result_merged_df["majority_answer_correct"],
    ) = zip(*pair_result_merged_df.apply(calc_accuracy, axis=1))

    pair_result_dfs[(dataset_name, model_w_params)] = pair_result_merged_df


In [ ]:
scores = []
for (dataset_name, model_w_params), pair_result_df in pair_result_dfs.items():
    abstention_rate_mean = pair_result_df["abstention_rate"].mean()
    factor = t.ppf(
        1 - 0.05 / 2, df=pair_result_df["abstention_rate"].count() - 1
    ) / np.sqrt(pair_result_df["abstention_rate"].count())
    abstention_rate_ci_half_width = pair_result_df["abstention_rate"].std() * factor

    overall_accuracy_mean = pair_result_df["overall_accuracy"].mean()
    factor = t.ppf(
        1 - 0.05 / 2, df=pair_result_df["overall_accuracy"].count() - 1
    ) / np.sqrt(pair_result_df["overall_accuracy"].count())
    overall_accuracy_ci_half_width = pair_result_df["overall_accuracy"].std() * factor

    answered_accuracy_mean = pair_result_df["answered_accuracy"].mean()
    factor = t.ppf(
        1 - 0.05 / 2, df=pair_result_df["answered_accuracy"].count() - 1
    ) / np.sqrt(pair_result_df["answered_accuracy"].count())
    answered_accuracy_ci_half_width = pair_result_df["answered_accuracy"].std() * factor

    scores.append(
        {
            "dataset_name": dataset_name,
            "model_w_params": model_w_params,
            "abstention_rate_mean": abstention_rate_mean,
            "abstention_rate_ci_half_width": abstention_rate_ci_half_width,
            "overall_accuracy_mean": overall_accuracy_mean,
            "overall_accuracy_ci_half_width": overall_accuracy_ci_half_width,
            "answered_accuracy_mean": answered_accuracy_mean,
            "answered_accuracy_ci_half_width": answered_accuracy_ci_half_width,
        }
    )
scores_df = pd.DataFrame(scores)

In [ ]:
# Plot scores with error bars
vars = [
    "abstention_rate_mean",
    "overall_accuracy_mean",
    "answered_accuracy_mean",
]
var_labels = {
    "abstention_rate_mean": "Mean abstention rate",
    "overall_accuracy_mean": "Mean overall accuracy",
    "answered_accuracy_mean": "Mean answered accuracy",
}
ncol = 1
nsub = len(vars)
nrow = nsub // ncol + (nsub % ncol > 0)
fig, axes = plt.subplots(nrow, ncol, figsize=(ncol * 5, nrow * 3), sharex=True)
axes = np.atleast_1d(axes).ravel()
handles, labels = None, None
for i, var in enumerate(vars):
    ax = plt.subplot(nrow, ncol, i + 1)

    x_labels = scores_df["dataset_name"].unique()
    models = scores_df["model_w_params"].unique()
    x = np.arange(len(x_labels))
    width = 0.8 / len(models)

    for j, model in enumerate(models):
        model_data = scores_df[scores_df["model_w_params"] == model]
        means = model_data[var].to_numpy()
        cis = model_data[var.replace("_mean", "_ci_half_width")].to_numpy()
        ax.bar(
            x + j * width,
            means,
            width=width,
            label=model,
            yerr=cis,
            capsize=3 if cis is not None else 0,
            alpha=0.8,
        )
    # ax.set_ylim(0.0, 1.0)
    # ax.set_yticks(np.arange(0.0, 1.01, 0.1))
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.tick_params(axis="y", labelsize=8)
    ax.set_ylabel(f"{var_labels[var]}", fontsize=10)
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels(x_labels, fontsize=8, rotation=90)
    ax.set_xlabel("", fontsize=10)
    if i == 0:
        handles, labels = ax.get_legend_handles_labels()
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
plt.figlegend(
    handles,
    labels,
    loc="center left",
    fontsize=10,
    bbox_to_anchor=(0.99, 0.5),
    frameon=False,
)
plt.tight_layout()
plt.show()

## Risk-coverage curve

In [ ]:
risk_coverage_curves = []
risk_coverage_curve_results = []
for (dataset_name, model_w_params), pair_result_df in pair_result_dfs.items():
    n_all = len(pair_result_df)

    valid_mask = (
        pair_result_df["majority_answer_correct"].notna()
        & pair_result_df["abstention_rate"].notna()
        & np.isfinite(pair_result_df["abstention_rate"])
    )

    tau_candidates = sorted(
        pair_result_df.loc[valid_mask, "abstention_rate"].unique().tolist()
    )

    curve = [
        {
            "tau": -np.inf,
            "risk": 0.0,
            "coverage": 0.0,
        }
    ]
    for tau in tau_candidates:
        accepted_df = pair_result_df[
            valid_mask & (pair_result_df["abstention_rate"] <= tau)
        ]
        n_accepted = len(accepted_df)

        risk = 1.0 - accepted_df["majority_answer_correct"].mean()
        coverage = n_accepted / n_all

        curve.append(
            {
                "tau": tau,
                "risk": risk,
                "coverage": coverage,
            }
        )

    # sort
    curve = sorted(curve, key=lambda x: x["coverage"])

    # remove duplicate coverage points
    deduped_curve = []
    seen_coverages = set()
    for point in curve:
        coverage_key = point["coverage"]
        if coverage_key not in seen_coverages:
            deduped_curve.append(point)
            seen_coverages.add(coverage_key)
    curve = deduped_curve

    # save risk-coverage curve points
    for point in curve:
        risk_coverage_curves.append(
            {
                "dataset_name": dataset_name,
                "model_w_params": model_w_params,
                **point,
            }
        )

    positive_points = [point for point in curve if point["coverage"] > 0]

    # max coverage
    max_coverage = max([point["coverage"] for point in positive_points], default=0.0)

    # min coverage
    min_coverage = min([point["coverage"] for point in positive_points], default=np.nan)

    # area under risk-coverage curve (step integral)
    if len(positive_points) == 0:
        aurcc_step_integral = np.nan
    else:
        aurcc_step_integral = 0.0
        for i in range(1, len(curve)):
            x1 = curve[i - 1]["coverage"]
            x2, y2 = curve[i]["coverage"], curve[i]["risk"]
            aurcc_step_integral += (x2 - x1) * y2

    # penalized area under risk-coverage curve (step integral)
    penalty_risks = [(0.5, "0p5"), (1.0, "1p0")]
    penalized_aurcc_step_integral_dict = {}
    for penalty_risk, penalty_risk_tag in penalty_risks:
        if len(positive_points) == 0:
            penalized_aurcc_step_integral = penalty_risk
        else:
            unattained_coverage = max(0.0, 1.0 - max_coverage)
            penalized_aurcc_step_integral = (
                aurcc_step_integral + penalty_risk * unattained_coverage
            )

        penalized_aurcc_step_integral_dict[penalty_risk_tag] = (
            penalized_aurcc_step_integral
        )

    # C@R
    coverage_at_risk_0p2 = max(
        [point["coverage"] for point in positive_points if point["risk"] <= 0.2],
        default=0.0,
    )

    # R@C
    risk_at_coverage_0p8 = np.nan
    for point in positive_points:
        if np.isnan(risk_at_coverage_0p8) and point["coverage"] >= 0.8:
            risk_at_coverage_0p8 = point["risk"]

    risk_coverage_curve_results.append(
        {
            "dataset_name": dataset_name,
            "model_w_params": model_w_params,
            "aurcc_step_integral": aurcc_step_integral,
            "penalized_aurcc_step_integral_0p5": penalized_aurcc_step_integral_dict[
                "0p5"
            ],
            "penalized_aurcc_step_integral_1p0": penalized_aurcc_step_integral_dict[
                "1p0"
            ],
            "max_coverage": max_coverage,
            "min_coverage": min_coverage,
            "coverage_at_risk_0p2": coverage_at_risk_0p2,
            "risk_at_coverage_0p8": risk_at_coverage_0p8,
        }
    )

risk_coverage_curves_df = pd.DataFrame(risk_coverage_curves)
risk_coverage_curve_results_df = pd.DataFrame(risk_coverage_curve_results)

In [ ]:
# Plot risk-coverage curve
dataset_names = [
    d
    for d in DATASET_NAME_ORDER
    if d in risk_coverage_curves_df["dataset_name"].unique()
]
n_sub = len(dataset_names)
n_col = 2
n_row = int(np.ceil(n_sub / n_col))
fig, axes = plt.subplots(n_row, n_col, figsize=(n_col * 4, n_row * 4))
axes = np.array(axes).reshape(-1)
handles, labels = None, None
for i, dataset_name in enumerate(dataset_names):
    ax = axes[i]
    dfi = risk_coverage_curves_df[
        risk_coverage_curves_df["dataset_name"] == dataset_name
    ]
    sns.lineplot(
        data=dfi,
        x="coverage",
        y="risk",
        hue="model_w_params",
        style="model_w_params",
        estimator=None,
        lw=2,
        markers=True,
        markersize=10,
        dashes=True,
        ax=ax,
    )
    ax.set_title(dataset_name, fontsize=10)
    ax.set_xlabel("Coverage", fontsize=10)
    ax.set_ylabel("Risk", fontsize=10)
    ax.set_xlim(0, 1)
    ax.set_xticks(np.arange(0.0, 1.01, 0.1))
    ax.set_ylim(0, 1)
    ax.set_yticks(np.arange(0.0, 1.01, 0.1))
    ax.tick_params(axis="x", labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(True, linestyle="--", alpha=0.3)
    if i == 0:
        handles, labels = ax.get_legend_handles_labels()
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()
for j in range(n_sub, len(axes)):
    axes[j].axis("off")
if handles is not None and labels is not None:
    fig.legend(
        handles,
        labels,
        loc="upper center",
        fontsize=10,
        bbox_to_anchor=(0.5, 1.06),
        frameon=True,
        ncol=3,
    )
plt.tight_layout()
plt.show()

In [ ]:
# Plot max coverage
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=risk_coverage_curve_results_df,
    x="dataset_name",
    y="max_coverage",
    hue="model_w_params",
)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Max coverage", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1, 0.5), frameon=False)
plt.show()

# Plot min coverage
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=risk_coverage_curve_results_df,
    x="dataset_name",
    y="min_coverage",
    hue="model_w_params",
)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Min coverage", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1, 0.5), frameon=False)
plt.show()

# Plot area under risk-coverage curve (step integral)
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=risk_coverage_curve_results_df,
    x="dataset_name",
    y="aurcc_step_integral",
    hue="model_w_params",
)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("AURC", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1, 0.5), frameon=False)
plt.show()

# Plot penalized area under risk-coverage curve (step integral)
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=risk_coverage_curve_results_df,
    x="dataset_name",
    y="penalized_aurcc_step_integral_0p5",
    hue="model_w_params",
)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Penalized AURC (penalty: 0.5)", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1, 0.5), frameon=False)
plt.show()

# Plot penalized area under risk-coverage curve (step integral)
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=risk_coverage_curve_results_df,
    x="dataset_name",
    y="penalized_aurcc_step_integral_1p0",
    hue="model_w_params",
)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("Penalized AURC (penalty: 1.0)", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1, 0.5), frameon=False)
plt.show()

# Plot coverage at risk 0.2
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=risk_coverage_curve_results_df,
    x="dataset_name",
    y="coverage_at_risk_0p2",
    hue="model_w_params",
)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("C@20%", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1, 0.5), frameon=False)
plt.show()

# Plot risk at coverage 0.8
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    data=risk_coverage_curve_results_df,
    x="dataset_name",
    y="risk_at_coverage_0p8",
    hue="model_w_params",
)
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.xticks(fontsize=10, rotation=90)
plt.yticks(fontsize=10)
plt.xlabel("", fontsize=12)
plt.ylabel("R@80%", fontsize=12)
plt.legend(loc="center left", fontsize=10, bbox_to_anchor=(1, 0.5), frameon=False)
plt.show()

In [ ]:
with open("results_final/pair_result_dfs.pkl", "wb") as f:
    pickle.dump(pair_result_dfs, f)

risk_coverage_curves_df.to_csv("results_final/risk_coverage_curves.csv", index=False)
risk_coverage_curve_results_df.to_csv(
    "results_final/risk_coverage_curve_results.csv", index=False
)